<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/processing_notebooks/02d_processing_nbhd_scatterplot_demographics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install contextily

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive
!ls /content/drive/Shareddrives

In [ ]:
import pandas as pd

gdf = gpd.read_file("/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/cleaned/open_close_after_2016.geojson")

In [ ]:
gdf.info()

In [ ]:
# # Importing SF geometry
# # URL for 2025 TIGER/Line Place boundaries - info here on
# ## how to use: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]


# project to same as our gdf
sf_poly = sf_poly.to_crs(epsg=4326)

## Neighborhood Level Analysis

In [ ]:
# Adding SF neighborhood geometry

nbhd_gdf = gpd.read_file(
    "/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/raw/sf_neighborhoods.geojson"
)

In [ ]:
nbhd_gdf.columns

In [ ]:
# Joining the gdf and nbhd gdf

gdf = gdf.set_crs(epsg=4326)
nbhd_gdf = nbhd_gdf.to_crs(epsg=4326)

# joining within
business_neighborhood_gdf = gpd.sjoin(gdf, nbhd_gdf, how="left", predicate="within")

In [ ]:
business_neighborhood_gdf.columns

In [ ]:
gdf = business_neighborhood_gdf.copy()

# filtering for 2019 to 2025
gdf = gdf[gdf['year'].between(2019, 2025)]

# openings
openings = (
    gdf[gdf['status'] == 'opened']
    .groupby(['nhood', 'year'])
    .size()
    .unstack(fill_value=0)
)

# closings
closings = (
    gdf[gdf['status'] == 'closed']
    .groupby(['nhood', 'year'])
    .size()
    .unstack(fill_value=0)
)

all_years = range(2019, 2026)
openings = openings.reindex(columns=all_years, fill_value=0)
closings = closings.reindex(columns=all_years, fill_value=0)

# renaming columns to be open_year
openings.columns = [f"open_{y}" for y in openings.columns]
closings.columns = [f"close_{y}" for y in closings.columns]

# combining openings and closings to get a table to compare across neighborhoods
neighborhood_table = openings.join(closings, how='outer').fillna(0).astype(int)

neighborhood_table

# Baseline by number of businesses in each neighborhood

In [ ]:
gdf = business_neighborhood_gdf.copy()

# baseline by total # of businesses
baseline = gdf.groupby('nhood')['uniqueid'].nunique()

# closures during COVID (2020–2021)
closures = gdf[
    (gdf['year'].between(2020, 2021)) &
    (gdf['status'] == 'closed')
].groupby('nhood').size()

# openings during recovery (2022–2025)
openings = gdf[
    (gdf['year'].between(2022, 2025)) &
    (gdf['status'] == 'opened')
].groupby('nhood').size()

# combine
rate_table = pd.DataFrame({
    'baseline': baseline,
    'closures_2020_21': closures,
    'openings_2022_25': openings
}).fillna(0)

# adding rates
rate_table['closure_rate'] = rate_table['closures_2020_21'] / rate_table['baseline']
rate_table['recovery_rate'] = rate_table['openings_2022_25'] / rate_table['baseline']

rate_table

## Calculate Business Resilience Rate

In [ ]:
# Filtering to nbhds with at least 50 businesses
rate_table_filtered = rate_table[rate_table['baseline'] >= 50]

In [ ]:
df = rate_table_filtered.copy()
df['resilience'] = df['recovery_rate'] - df['closure_rate']

top_10 = df.sort_values('resilience', ascending=False).head(10)
bottom_10 = df.sort_values('resilience', ascending=True).head(10)

print("Most Resilient Neighborhoods")
display(top_10)

print("Least Resilient Neighborhoods")
display(bottom_10)

# Demographic Data by Neighborhood

In [ ]:
import geopandas as gpd
from shapely import wkt

# Read in tract to neighborhood geometry file
neighborhoods = pd.read_csv('/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/raw/tract_to_neighborhood_geom.csv')

# Convert WKT geometry column to shapely geometry and make GeoDataFrame
neighborhoods["geometry"] = neighborhoods["the_geom"].apply(wkt.loads)
neighborhoods = gpd.GeoDataFrame(neighborhoods, geometry="geometry")

In [ ]:
neighborhoods = neighborhoods[[
    "geoid",
    "neighborhoods_analysis_boundaries",
    "geometry"
]].rename(columns={
    "geoid": "tract",
    "neighborhoods_analysis_boundaries": "neighborhood"
})

In [ ]:
neighborhoods.head()

In [ ]:
!pip install census
from census import Census
import pandas as pd

# Prepare Census Bureau API for data pulling
api_key = 'c9ad2fda3c683e4a14992e89daed0afee55738d2'
c = Census(key=api_key)

In [ ]:
race_vars = {
    "B03002_001E": "population",
    "B03002_003E": "white",           # Non-Hispanic White
    "B03002_004E": "black",           # Non-Hispanic Black
    "B03002_005E": "aian",            # Non-Hispanic American Indian/Alaska Native
    "B03002_006E": "asian",           # Non-Hispanic Asian
    "B03002_007E": "nhpi",            # Non-Hispanic Native Hawaiian/Pacific Islander
    "B03002_008E": "other",           # Non-Hispanic Some Other Race
    "B03002_012E": "latina_o",        # Hispanic or Latino
}

income_vars = {
    "B19013_001E": "median_income"
}

all_vars = {**race_vars, **income_vars}

# Pulling ACS 5-year 2023 data
acs = pd.DataFrame(
    c.acs5.get(
        list(all_vars.keys()),
        {'for': 'tract:*', 'in': 'state:06 county:075'},
        year=2023
    )
).rename(columns=all_vars)

acs['tract'] = '06075' + acs['tract']

In [ ]:
# Replace Census null codes with NaN
acs['median_income'] = acs['median_income'].replace(-666666666, pd.NA)

In [ ]:
acs.head()

In [ ]:
neighborhoods.head()

In [ ]:
acs['tract'] = acs['tract'].astype(str).str.lstrip('0')

neighborhoods['tract'] = neighborhoods['tract'].astype(str)
acs['tract'] = acs['tract'].astype(str)

merged = neighborhoods.merge(acs, on='tract', how='inner')

# Merge with neighborhoods shapefile
merged = neighborhoods.merge(acs, on='tract', how='inner')

# Aggregate by neighborhood
agg_cols = {
    'population': 'sum',
    'white': 'sum',
    'black': 'sum',
    'aian': 'sum',
    'asian': 'sum',
    'nhpi': 'sum',
    'other': 'sum',
    'latina_o': 'sum',
    'median_income': 'mean'
}

final = merged.groupby('neighborhood').agg(agg_cols).reset_index()



In [ ]:
# Percentages
# combining Native American into other because all the values are less than 1%
# combining NHPI with Asian because all the values are less than .02%
final['other'] = final['other'] + final['aian']
final = final.drop(columns='aian')
final['asian'] = final['asian'] + final['nhpi']
final = final.drop(columns='nhpi')

final['pct_other']   = final['other']   / final['population']
final['pct_white']   = final['white']   / final['population']
final['pct_black']   = final['black']   / final['population']
final['pct_asian']   = final['asian']   / final['population']
final['pct_latina_o']= final['latina_o']/ final['population']

# Filter out very small nbhds
final_df = final[final['population'] >= 200]

final_df

In [ ]:
# Set up to plot resilience rate by demographic variables

df = rate_table_filtered.copy()
df['resilience'] = df['recovery_rate'] - df['closure_rate']

# Merge in demographics
final_indexed = final.set_index('neighborhood')
final_indexed['pct_poc'] = 1 - final_indexed['pct_white']
demo_cols = ['median_income', 'pct_poc']
df = df.join(final_indexed[demo_cols], how='left')
df['median_income'] = pd.to_numeric(df['median_income'], errors='coerce')

df = df.sort_values('resilience', ascending=False)


In [ ]:
df.head()

In [ ]:
# Setting up file to download csv to use for visualizations

#Export file 1
# Keep only pct columns  and neighborhood info from final_df
pct_cols = ['neighborhood', 'population', 'median_income',
            'pct_white', 'pct_black', 'pct_asian', 'pct_latina_o', 'pct_other']
final_slim = final_df[pct_cols].set_index('neighborhood')

# Merge with df (which is indexed by nhood)
# Rename index to match
df.index.name = 'neighborhood'

combined = df.merge(final_slim, left_index=True, right_index=True, how='inner')

# Drop duplicate median_income column
combined = combined.drop(columns='median_income_y').rename(columns={'median_income_x': 'median_income'})

combined.to_csv('sf_business_demographics_nhood.csv')

print(combined.shape)
combined.head()

In [ ]:
gdf.head()

In [ ]:
# Export File 2: neighborhood_table with per-year open/close counts

open_close_cols = [col for col in neighborhood_table.columns if col.startswith(('open_', 'close_'))]
neighborhood_table[open_close_cols] = neighborhood_table[open_close_cols].fillna(0).astype(int)

neighborhood_table.to_csv('neighborhood_table_with_years.csv')

neighborhood_table = neighborhood_table.join(combined[['median_income', 'pct_poc']], how='left')
neighborhood_table.to_csv('neighborhood_table_with_years.csv', index=True)


# Export file 3: yearly counts by status (for the line chart)
yearly_counts = (
    gdf[gdf['year'] <= 2025]
    .groupby(['year', 'status'])
    .size()
    .reset_index(name='count')
)
yearly_counts.to_csv('netchange_yearly.csv', index=False)

In [ ]:
yearly_counts.head()

In [ ]:
from google.colab import files
files.download('sf_business_demographics_nhood.csv')
files.download('neighborhood_table_with_years.csv')
files.download('netchange_yearly.csv')